<a href="https://colab.research.google.com/github/MarquiseRosier/pi05-run/blob/main/notebooks/pi05_libero_colab_l4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pi0.5 LIBERO Colab L4 Notebook

This notebook runs `lerobot/pi05_libero_finetuned` on LIBERO in Google Colab without Docker.

Target workflow:

1. Open this notebook in Colab.
2. Runtime -> Change runtime type -> GPU -> choose **L4** when available.
3. Mount a restricted Google Drive folder containing model/assets cache archives and experiment outputs.
4. Runtime -> Run all.
5. Inspect rollout videos and activation diagnostics inline.

Access policy to apply in Google Drive/Colab:

- Share this notebook only with approved Google accounts.
- Current requested approved account: `programmer908@gmail.com`.
- Share the Drive cache/output folder only with the same approved accounts.
- Do not store Hugging Face tokens in the notebook or shared Drive folder.
- Put `HF_TOKEN` only in each user's private Colab Secrets if an authenticated cache refresh is needed.

Performance note: Google Drive is slow for huge model caches with many files. This notebook uses tar archives under `DRIVE_ROOT/archives/` and extracts them to `/content` local disk before running.

Codex cannot enforce Google Drive sharing ACLs from this repo. Apply those restrictions in the Colab/Drive UI after saving the notebook copy.

## What This Runs

The standard benchmark path uses LeRobot eval:

```text
LIBERO simulator -> 2 camera tensors + robot state + task text -> Pi0.5 -> [50, 7] action chunk -> LIBERO simulator
```

The model receives two meaningful camera views on each policy call:

```text
observation.images.image   # agent view
observation.images.image2  # wrist / eye-in-hand view
```

It predicts 50 future actions. LeRobot executes the first 10 actions, then gets fresh camera/state observations and replans.

The display section creates an inline report for mechanistic interpretability: rollout video, four-panel diagnostic video, chunk matrix, family activation heatmaps, and per-expert-layer activation graphs. The final section includes an experimental single-chunk prompt probe. Benchmark scoring still uses official LIBERO task prompts and success predicates.

In [ ]:
# @title Controls

# Drive folder shared only with approved collaborators.
DRIVE_ROOT = "/content/drive/MyDrive/groot-run-shared-programmer908"  # @param {type:"string"}
APPROVED_ACCOUNTS = "programmer908@gmail.com"  # @param {type:"string"}

# Repository.
REPO_URL = "https://github.com/MarquiseRosier/pi05-run.git"  # @param {type:"string"}
REPO_BRANCH = "main"  # @param {type:"string"}

# Runtime preference. Use Any if Colab allocated a T4; set L4/A100 to enforce that GPU.
REQUIRED_GPU = "Any"  # @param ["L4", "A100", "Any"]

# Evaluation controls.
SUITE = "libero_spatial"  # @param ["libero_spatial", "libero_object", "libero_goal", "libero_10", "libero_spatial,libero_object,libero_goal,libero_10"]
TASK_IDS = "[0]"  # @param {type:"string"}
ANALYSIS_TASK_ID = 0  # @param {type:"integer"}
EPISODES = 1  # @param {type:"integer"}
EVAL_PROGRESS_SECONDS = 30  # @param {type:"integer"}

# Activation capture controls.
CAPTURE_ACTIVATIONS = True  # @param {type:"boolean"}
CAPTURE_PARAM_STATS = False  # @param {type:"boolean"}
CAPTURE_MAX_CHUNKS = 40  # @param {type:"integer"}
CAPTURE_LAYER_STRIDE = 1  # @param {type:"integer"}
CAPTURE_MAX_BINS = 64  # @param {type:"integer"}

# Colab report controls.
REPORT_MAX_ROWS = 80  # @param {type:"integer"}
GENERATE_DIAGNOSTIC_VIDEO = False  # @param {type:"boolean"}
DISPLAY_INDIVIDUAL_LAYER_GRAPHS = False  # @param {type:"boolean"}
LAYER_GRAPH_LIMIT = 6  # @param {type:"integer"}

# Cache behavior. Archive mode avoids slow Google Drive small-file copies.
CACHE_TRANSFER_MODE = "archive"  # @param ["archive", "folders"]
ALLOW_AUTH_REFRESH = True  # @param {type:"boolean"}
FORCE_AUTH_REFRESH = False  # @param {type:"boolean"}
HF_OFFLINE = True  # @param {type:"boolean"}

# Experimental prompt probe controls.
PROBE_SUITE = "libero_spatial"  # @param ["libero_spatial", "libero_object", "libero_goal", "libero_10"]
PROBE_TASK_ID = 0  # @param {type:"integer"}
PROBE_LANGUAGE = "pick up the black bowl between the plate and the ramekin and place it on the plate"  # @param {type:"string"}
PROBE_SEED = 1000  # @param {type:"integer"}

print("Configured approved accounts:", APPROVED_ACCOUNTS)


In [ ]:
# @title Mount Drive And Validate Runtime

from pathlib import Path
import os
import subprocess

from google.colab import drive

drive.mount("/content/drive")

DRIVE_ROOT = Path(DRIVE_ROOT)
DRIVE_ARCHIVES = DRIVE_ROOT / "archives"
DRIVE_HF_HOME = DRIVE_ROOT / "hf_home"
DRIVE_LIBERO_CACHE = DRIVE_ROOT / "libero_cache"
DRIVE_LIBERO_DATASETS = DRIVE_ROOT / "libero_datasets"
DRIVE_OUTPUTS = DRIVE_ROOT / "outputs"
DRIVE_NOTEBOOK_META = DRIVE_ROOT / "notebook_meta"
for path in [DRIVE_ROOT, DRIVE_ARCHIVES, DRIVE_HF_HOME, DRIVE_LIBERO_CACHE, DRIVE_LIBERO_DATASETS, DRIVE_OUTPUTS, DRIVE_NOTEBOOK_META]:
    path.mkdir(parents=True, exist_ok=True)

HF_HOME_ARCHIVE = DRIVE_ARCHIVES / "hf_home.tar"
LIBERO_CACHE_ARCHIVE = DRIVE_ARCHIVES / "libero_cache.tar"
LIBERO_DATASETS_ARCHIVE = DRIVE_ARCHIVES / "libero_datasets.tar"

print("Drive root:", DRIVE_ROOT)
print("Archive dir:", DRIVE_ARCHIVES)
print("Expected sharing: restricted to", APPROVED_ACCOUNTS)

result = subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"], text=True, capture_output=True)
print(result.stdout or result.stderr)
if result.returncode != 0:
    raise RuntimeError("No NVIDIA GPU visible. In Colab: Runtime -> Change runtime type -> GPU.")

gpu_line = result.stdout.strip().splitlines()[0]
if REQUIRED_GPU != "Any" and REQUIRED_GPU.lower() not in gpu_line.lower():
    raise RuntimeError(f"Requested {REQUIRED_GPU}, but Colab allocated: {gpu_line}. Change runtime or set REQUIRED_GPU='Any'.")

In [ ]:
# @title Install Native Runtime Equivalent To cloud/libero/Dockerfile

import os
import subprocess
from pathlib import Path

VENV = Path("/content/lerobot-venv")
PYTHON = VENV / "bin/python"
UV_BIN_DIR = Path("/content/uv-bin")
UV = str(UV_BIN_DIR / "uv")


def run(cmd, *, env=None):
    cmd = list(map(str, cmd))
    print("$", " ".join(cmd), flush=True)
    try:
        return subprocess.run(cmd, env=env, check=True)
    except subprocess.CalledProcessError as exc:
        print(f"Command failed with exit code {exc.returncode}: {' '.join(cmd)}", flush=True)
        raise


def install_uv() -> str:
    # Colab may already have /usr/local/bin/uv, but its version/behavior can differ.
    # Use a notebook-owned binary so every clean runtime follows the same path.
    UV_BIN_DIR.mkdir(parents=True, exist_ok=True)
    installer = Path("/tmp/install-uv.sh")
    run(["curl", "-LsSf", "https://astral.sh/uv/install.sh", "-o", installer])
    env = os.environ.copy()
    env["UV_INSTALL_DIR"] = str(UV_BIN_DIR)
    run(["sh", installer], env=env)
    os.environ["PATH"] = f"{UV_BIN_DIR}:" + os.environ["PATH"]
    run([UV, "--version"])
    return UV


apt_packages = [
    "build-essential", "ca-certificates", "cmake", "curl", "ffmpeg", "git", "pv", "rsync",
    "libegl1", "libgl1", "libglib2.0-0", "libglvnd0", "libglx0", "libopengl0",
    "libosmesa6-dev", "libsm6", "libxext6", "libxrender1", "pkg-config",
]
apt_env = os.environ.copy()
apt_env["DEBIAN_FRONTEND"] = "noninteractive"
run(["apt-get", "update", "-qq"], env=apt_env)
run(["apt-get", "install", "-y", "-qq", *apt_packages], env=apt_env)
UV = install_uv()

# Managed Python avoids system-package/venv quirks in Colab images.
run([UV, "python", "install", "3.12"])
run([UV, "venv", "--clear", str(VENV), "--python", "3.12"])

run([
    UV, "pip", "install", "--python", str(PYTHON), "--torch-backend", "cu128",
    "lerobot[evaluation,libero,pi]", "hf-transfer", "opencv-python", "numpy",
])

os.environ["PATH"] = f"{VENV / 'bin'}:{UV_BIN_DIR}:" + os.environ["PATH"]
print("Python:", PYTHON)
run([str(PYTHON), "-c", "import torch; print('torch', torch.__version__); print('cuda', torch.cuda.is_available()); print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no gpu')"])


In [ ]:
# @title Clone Or Update Repo

from pathlib import Path
import subprocess

LOCAL_REPO = Path("/content/groot-run")
if LOCAL_REPO.exists():
    subprocess.run(["git", "-C", str(LOCAL_REPO), "fetch", "origin", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", str(LOCAL_REPO), "checkout", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", str(LOCAL_REPO), "pull", "--ff-only", "origin", REPO_BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(LOCAL_REPO)], check=True)

print("Repo:", LOCAL_REPO)
subprocess.run(["git", "-C", str(LOCAL_REPO), "rev-parse", "--short", "HEAD"], check=True)

In [ ]:
# @title Prepare Drive Caches And Offline Mode

from pathlib import Path
import os
import shutil
import shlex
import subprocess
import time

LOCAL_HF_HOME = Path("/content/hf_home")
LOCAL_LIBERO_CACHE = Path.home() / ".cache/libero"
LOCAL_OUTPUT_ROOT = LOCAL_REPO / "outputs/eval/pi05_libero"
LOCAL_DATA_ROOT = LOCAL_REPO / "data/libero/datasets"
LOCAL_LIBERO_CONFIG = LOCAL_REPO / ".libero"
for path in [LOCAL_OUTPUT_ROOT, LOCAL_LIBERO_CONFIG]:
    path.mkdir(parents=True, exist_ok=True)

repos = ["lerobot/pi05_libero_finetuned", "google/paligemma-3b-pt-224"]


def now_text() -> str:
    return time.strftime("%H:%M:%S")


def format_duration(seconds: float | None) -> str:
    if seconds is None or seconds != seconds or seconds < 0:
        return "unknown"
    seconds = int(seconds)
    hours, rem = divmod(seconds, 3600)
    minutes, secs = divmod(rem, 60)
    if hours:
        return f"{hours}h{minutes:02d}m{secs:02d}s"
    if minutes:
        return f"{minutes}m{secs:02d}s"
    return f"{secs}s"


def format_bytes(value: float | int | None) -> str:
    if value is None:
        return "unknown"
    value = float(value)
    units = ["B", "KiB", "MiB", "GiB", "TiB"]
    for unit in units:
        if abs(value) < 1024 or unit == units[-1]:
            return f"{value:.1f}{unit}" if unit != "B" else f"{value:.0f}{unit}"
        value /= 1024
    return f"{value:.1f}PiB"


def timed(label, fn):
    print(f"\n== {label} | start {now_text()} ==", flush=True)
    start = time.time()
    result = fn()
    elapsed = time.time() - start
    print(f"== done: {label} | elapsed {format_duration(elapsed)} | finish {now_text()} ==", flush=True)
    return result


def run(cmd, **kwargs):
    print("+", " ".join(map(str, cmd)), flush=True)
    start = time.time()
    result = subprocess.run([str(x) for x in cmd], check=True, **kwargs)
    print(f"+ done in {format_duration(time.time() - start)}", flush=True)
    return result


def reset_dir(path: Path):
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)


def dir_has_anything(path: Path) -> bool:
    return path.exists() and any(path.iterdir())


def hf_cache_has(repo_id: str, root: Path) -> bool:
    namespace, name = repo_id.split("/", 1)
    return (root / "hub" / f"models--{namespace}--{name}").exists()


def path_size_bytes(path: Path) -> int:
    if not path.exists():
        return 0
    total = 0
    for root, dirs, files in os.walk(path):
        dirs[:] = [name for name in dirs if not Path(root, name).is_symlink()]
        for name in files:
            file_path = Path(root, name)
            try:
                total += file_path.lstat().st_size
            except OSError:
                pass
    return total


def du(path: Path):
    if path.exists():
        run(["du", "-sh", path])


def extract_tar(archive: Path, dst: Path):
    reset_dir(dst)
    size = archive.stat().st_size
    print(f"Archive: {archive} ({format_bytes(size)}) -> {dst}", flush=True)
    cmd = f"set -euo pipefail; pv -ptebarf -s {size} {shlex.quote(str(archive))} | tar -C {shlex.quote(str(dst))} -xf -"
    run(["bash", "-lc", cmd])
    du(dst)


def create_tar(src: Path, archive: Path):
    if not dir_has_anything(src):
        print(f"Skipping archive for empty directory: {src}", flush=True)
        return
    archive.parent.mkdir(parents=True, exist_ok=True)
    tmp = archive.with_suffix(archive.suffix + ".tmp")
    if tmp.exists():
        tmp.unlink()
    size_bytes = path_size_bytes(src)
    print(f"Archiving: {src} ({format_bytes(size_bytes)}) -> {archive}", flush=True)
    cmd = f"set -euo pipefail; tar -C {shlex.quote(str(src))} -cf - . | pv -ptebarf -s {size_bytes} > {shlex.quote(str(tmp))}"
    run(["bash", "-lc", cmd])
    tmp.replace(archive)
    run(["ls", "-lh", archive])


def rsync_tree(src: Path, dst: Path, reset: bool = True):
    src.mkdir(parents=True, exist_ok=True)
    if reset:
        reset_dir(dst)
    else:
        dst.mkdir(parents=True, exist_ok=True)
    print(f"Rsync: {src} ({format_bytes(path_size_bytes(src))}) -> {dst}", flush=True)
    run(["rsync", "-a", "--human-readable", "--info=progress2", "--stats", f"{src}/", f"{dst}/"])
    du(dst)


if CACHE_TRANSFER_MODE == "archive" and HF_HOME_ARCHIVE.exists() and not FORCE_AUTH_REFRESH:
    timed("extract HF cache archive from Drive to /content", lambda: extract_tar(HF_HOME_ARCHIVE, LOCAL_HF_HOME))
elif ALLOW_AUTH_REFRESH:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
    if not token:
        raise RuntimeError("ALLOW_AUTH_REFRESH=True requires a private Colab Secret named HF_TOKEN, unless DRIVE_ROOT/archives/hf_home.tar already exists.")
    reset_dir(LOCAL_HF_HOME)
    run([str(PYTHON), "-c", "import huggingface_hub; print('huggingface_hub OK')"])
    refresh_code = """
from huggingface_hub import HfApi, snapshot_download
from huggingface_hub.utils import enable_progress_bars
import os
import subprocess
import threading
import time

repos = os.environ['REFRESH_REPOS'].split(',')
token = os.environ.get('HF_TOKEN') or None
cache_dir = os.environ['LOCAL_HF_HUB_CACHE']
heartbeat_seconds = int(os.environ.get('HF_PROGRESS_HEARTBEAT_SECONDS', '15'))


def now_text():
    return time.strftime('%H:%M:%S')


def format_duration(seconds):
    if seconds is None or seconds != seconds or seconds < 0:
        return 'unknown'
    seconds = int(seconds)
    hours, rem = divmod(seconds, 3600)
    minutes, secs = divmod(rem, 60)
    if hours:
        return f'{hours}h{minutes:02d}m{secs:02d}s'
    if minutes:
        return f'{minutes}m{secs:02d}s'
    return f'{secs}s'


def format_bytes(value):
    if value is None:
        return 'unknown'
    value = float(value)
    units = ['B', 'KiB', 'MiB', 'GiB', 'TiB']
    for unit in units:
        if abs(value) < 1024 or unit == units[-1]:
            return f'{value:.1f}{unit}' if unit != 'B' else f'{value:.0f}{unit}'
        value /= 1024
    return f'{value:.1f}PiB'


def path_size_bytes(path):
    if not os.path.exists(path):
        return 0
    total = 0
    for root, dirs, files in os.walk(path):
        dirs[:] = [name for name in dirs if not os.path.islink(os.path.join(root, name))]
        for name in files:
            file_path = os.path.join(root, name)
            try:
                total += os.lstat(file_path).st_size
            except OSError:
                pass
    return total


def estimate_repo_size(api, repo):
    try:
        info = api.model_info(repo_id=repo, token=token, files_metadata=True)
        sizes = [getattr(sibling, 'size', None) for sibling in getattr(info, 'siblings', [])]
        total = sum(size for size in sizes if isinstance(size, int))
        count = len(sizes)
        return count, total or None
    except Exception as exc:
        print(f'[{repo}] could not estimate size before download: {type(exc).__name__}: {exc}', flush=True)
        return None, None


def heartbeat(repo, root, start_bytes, estimated_total, stop_event):
    last_time = time.time()
    last_bytes = start_bytes
    start_time = last_time
    while not stop_event.wait(heartbeat_seconds):
        now = time.time()
        current_bytes = path_size_bytes(root)
        delta = max(0, current_bytes - start_bytes)
        recent_rate = max(0, current_bytes - last_bytes) / max(0.001, now - last_time)
        avg_rate = delta / max(0.001, now - start_time)
        if estimated_total and avg_rate > 0:
            remaining = max(0, estimated_total - delta)
            eta = remaining / avg_rate
            pct = min(100.0, (delta / estimated_total) * 100.0)
            estimate_text = f'{pct:5.1f}% of est. {format_bytes(estimated_total)} | ETA {format_duration(eta)}'
        else:
            estimate_text = 'ETA unknown'
        print(
            f'[{repo}] progress {format_bytes(delta)} | recent {format_bytes(recent_rate)}/s | '
            f'avg {format_bytes(avg_rate)}/s | elapsed {format_duration(now - start_time)} | {estimate_text}',
            flush=True,
        )
        last_time = now
        last_bytes = current_bytes


os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '0'
os.environ['TQDM_DISABLE'] = '0'
os.environ['TQDM_MININTERVAL'] = '1'
try:
    enable_progress_bars()
except Exception as exc:
    print(f'Could not force-enable Hugging Face progress bars: {type(exc).__name__}: {exc}', flush=True)

api = HfApi()
for index, repo in enumerate(repos, 1):
    print(f'\\n[{index}/{len(repos)}] preparing {repo} at {now_text()}', flush=True)
    file_count, estimated_total = estimate_repo_size(api, repo)
    if file_count is not None:
        print(f'[{repo}] estimated remote files={file_count} size={format_bytes(estimated_total)}', flush=True)
    before = path_size_bytes(cache_dir)
    stop_event = threading.Event()
    monitor = threading.Thread(target=heartbeat, args=(repo, cache_dir, before, estimated_total, stop_event), daemon=True)
    start = time.time()
    monitor.start()
    try:
        path = snapshot_download(
            repo_id=repo,
            cache_dir=cache_dir,
            token=token,
            max_workers=8,
        )
    finally:
        stop_event.set()
        monitor.join(timeout=2)
    elapsed = time.time() - start
    after = path_size_bytes(cache_dir)
    delta = max(0, after - before)
    avg_rate = delta / max(0.001, elapsed)
    print(f'[{repo}] -> {path}', flush=True)
    print(
        f'[{repo}] done at {now_text()} | elapsed {format_duration(elapsed)} | '
        f'cache delta {format_bytes(delta)} | avg {format_bytes(avg_rate)}/s | total cache {format_bytes(after)}',
        flush=True,
    )
"""
    env = os.environ.copy()
    for key in ["HF_HUB_OFFLINE", "TRANSFORMERS_OFFLINE", "HF_DATASETS_OFFLINE"]:
        env.pop(key, None)
    env.update({
        "PYTHONUNBUFFERED": "1",
        "HF_TOKEN": token,
        "HF_HOME": str(LOCAL_HF_HOME),
        "HF_HUB_CACHE": str(LOCAL_HF_HOME / "hub"),
        "LOCAL_HF_HUB_CACHE": str(LOCAL_HF_HOME / "hub"),
        "REFRESH_REPOS": ",".join(repos),
        "HF_HUB_ENABLE_HF_TRANSFER": "1",
        "HF_XET_HIGH_PERFORMANCE": "1",
        "HF_HUB_DISABLE_PROGRESS_BARS": "0",
        "TQDM_DISABLE": "0",
        "TQDM_MININTERVAL": "1",
        "HF_PROGRESS_HEARTBEAT_SECONDS": "15",
    })
    timed("download gated HF models to fast local disk", lambda: subprocess.run([str(PYTHON), "-u", "-c", refresh_code], env=env, check=True))
elif CACHE_TRANSFER_MODE == "folders" or dir_has_anything(DRIVE_HF_HOME):
    print("Archive missing; falling back to Drive folder rsync. This can be very slow for HF caches.", flush=True)
    timed("copy HF cache folder from Drive to /content", lambda: rsync_tree(DRIVE_HF_HOME, LOCAL_HF_HOME))
else:
    raise RuntimeError(
        "No HF cache archive or folder found in Drive. Set ALLOW_AUTH_REFRESH=True once with private Colab Secret HF_TOKEN."
    )

missing_local = [repo for repo in repos if not hf_cache_has(repo, LOCAL_HF_HOME)]
if missing_local:
    raise RuntimeError(
        "Local HF cache is missing: " + ", ".join(missing_local) + "\n"
        "Run once with ALLOW_AUTH_REFRESH=True, or populate DRIVE_ROOT/archives/hf_home.tar."
    )

if CACHE_TRANSFER_MODE == "archive" and LIBERO_CACHE_ARCHIVE.exists():
    timed("extract LIBERO cache archive from Drive", lambda: extract_tar(LIBERO_CACHE_ARCHIVE, LOCAL_LIBERO_CACHE))
else:
    timed("copy LIBERO cache folder from Drive", lambda: rsync_tree(DRIVE_LIBERO_CACHE, LOCAL_LIBERO_CACHE))

if CACHE_TRANSFER_MODE == "archive" and LIBERO_DATASETS_ARCHIVE.exists():
    timed("extract LIBERO datasets archive from Drive", lambda: extract_tar(LIBERO_DATASETS_ARCHIVE, LOCAL_DATA_ROOT))
else:
    timed("copy LIBERO datasets folder from Drive", lambda: rsync_tree(DRIVE_LIBERO_DATASETS, LOCAL_DATA_ROOT))

os.environ.update({
    "PATH": f"{VENV / 'bin'}:" + os.environ["PATH"],
    "PYTHON": str(PYTHON),
    "HF_HOME": str(LOCAL_HF_HOME),
    "HF_HUB_CACHE": str(LOCAL_HF_HOME / "hub"),
    "HF_HUB_ENABLE_HF_TRANSFER": "1",
    "HF_XET_HIGH_PERFORMANCE": "1",
    "HF_HUB_DISABLE_PROGRESS_BARS": "0",
    "TQDM_DISABLE": "0",
    "TQDM_MININTERVAL": "1",
    "MUJOCO_GL": "egl",
    "PYOPENGL_PLATFORM": "egl",
    "MUJOCO_EGL_DEVICE_ID": "0",
    "MPLBACKEND": "Agg",
    "LIBERO_CONFIG_PATH": str(LOCAL_LIBERO_CONFIG),
    "LIBERO_DATASET_DIR": str(LOCAL_DATA_ROOT),
    "OUTPUT_ROOT": str(LOCAL_OUTPUT_ROOT),
})
if HF_OFFLINE:
    os.environ.update({"HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1", "HF_DATASETS_OFFLINE": "1"})
else:
    os.environ.pop("HF_HUB_OFFLINE", None)
    os.environ.pop("TRANSFORMERS_OFFLINE", None)
    os.environ.pop("HF_DATASETS_OFFLINE", None)

validate_code = """
from huggingface_hub import snapshot_download
import os
for repo in os.environ['VALIDATE_REPOS'].split(','):
    path = snapshot_download(repo_id=repo, cache_dir=os.environ['HF_HUB_CACHE'], local_files_only=True)
    print(repo, '->', path, flush=True)
"""
env = os.environ.copy()
env["VALIDATE_REPOS"] = ",".join(repos)
timed("validate local HF cache with local_files_only=True", lambda: subprocess.run([str(PYTHON), "-u", "-c", validate_code], env=env, check=True))

print("\nLocal cache summary:")
du(LOCAL_HF_HOME)
du(LOCAL_LIBERO_CACHE)
du(LOCAL_DATA_ROOT)


In [ ]:
# @title Run LIBERO Eval With Progress

import ast
import json
import os
import re
import subprocess
import sys
import threading
import time
from pathlib import Path

run_env = os.environ.copy()
run_id = time.strftime("%Y%m%d-%H%M%S", time.gmtime())
run_dir = LOCAL_OUTPUT_ROOT / run_id
suffix = 1
while run_dir.exists():
    run_id = f"{run_id}-{suffix}"
    run_dir = LOCAL_OUTPUT_ROOT / run_id
    suffix += 1
run_dir.mkdir(parents=True, exist_ok=True)

run_env.update({
    "PYTHON": str(PYTHON),
    "DEVICE": "cuda",
    "DTYPE": "bfloat16",
    "TASK_IDS": TASK_IDS.strip(),
    "RUN_ID": run_id,
    "OUTPUT_ROOT": str(LOCAL_OUTPUT_ROOT),
    "CAPTURE_ACTIVATIONS": "1" if CAPTURE_ACTIVATIONS else "0",
    "CAPTURE_PARAM_STATS": "1" if CAPTURE_PARAM_STATS else "0",
    "CAPTURE_MAX_CHUNKS": str(CAPTURE_MAX_CHUNKS),
    "CAPTURE_LAYER_STRIDE": str(CAPTURE_LAYER_STRIDE),
    "CAPTURE_MAX_BINS": str(CAPTURE_MAX_BINS),
    "PYTHONUNBUFFERED": "1",
    "HF_HUB_DISABLE_PROGRESS_BARS": "0",
    "TQDM_DISABLE": "0",
    "TQDM_MININTERVAL": "1",
    "MPLBACKEND": "Agg",
})

cmd = ["bash", "cloud/libero/run_pi05_libero.sh", SUITE, str(EPISODES)]
launcher_log = run_dir / "colab_launcher.log"

ANSI_RE = re.compile(r"\x1b\[[0-?]*[ -/]*[@-~]")


def eval_format_duration(seconds: float | None) -> str:
    if seconds is None or seconds != seconds or seconds < 0:
        return "unknown"
    seconds = int(seconds)
    hours, rem = divmod(seconds, 3600)
    minutes, secs = divmod(rem, 60)
    if hours:
        return f"{hours}h{minutes:02d}m{secs:02d}s"
    if minutes:
        return f"{minutes}m{secs:02d}s"
    return f"{secs}s"


def eval_format_bytes(value: float | int | None) -> str:
    if value is None:
        return "unknown"
    value = float(value)
    units = ["B", "KiB", "MiB", "GiB", "TiB"]
    for unit in units:
        if abs(value) < 1024 or unit == units[-1]:
            return f"{value:.1f}{unit}" if unit != "B" else f"{value:.0f}{unit}"
        value /= 1024
    return f"{value:.1f}PiB"


def eval_path_size_bytes(path: Path) -> int:
    if not path.exists():
        return 0
    total = 0
    for root, dirs, files in os.walk(path):
        dirs[:] = [name for name in dirs if not Path(root, name).is_symlink()]
        for name in files:
            file_path = Path(root, name)
            try:
                total += file_path.lstat().st_size
            except OSError:
                pass
    return total


def clean_log_text(text: str) -> str:
    return ANSI_RE.sub("", text).replace("\r", "\n")


def tail_text(path: Path, max_bytes: int = 240_000) -> str:
    if not path.exists():
        return ""
    with path.open("rb") as handle:
        try:
            handle.seek(-max_bytes, os.SEEK_END)
        except OSError:
            handle.seek(0)
        return handle.read().decode("utf-8", errors="replace")


def task_id_count(raw: str) -> int | None:
    raw = raw.strip()
    if not raw:
        return None
    try:
        value = ast.literal_eval(raw)
    except Exception:
        return None
    if isinstance(value, int):
        return 1
    if isinstance(value, (list, tuple, set)):
        return len(value)
    return None


def expected_eval_count() -> int:
    suites = [item.strip() for item in SUITE.split(",") if item.strip()]
    per_suite = task_id_count(TASK_IDS)
    if per_suite is None:
        per_suite = 10
    return max(1, len(suites) * max(1, per_suite) * max(1, int(EPISODES)))


def parse_progress_line(text: str) -> dict[str, str]:
    clean = clean_log_text(text)
    result: dict[str, str] = {}
    try:
        rollout_matches = list(re.finditer(
            r"Running rollout with at most\s+(\d+)\s+steps:\s+(\d+)%.*?\|\s*(\d+)/(?:\s*)?(\d+).*?running_success_rate=([0-9.]+)%",
            clean,
        ))
        if rollout_matches:
            match = rollout_matches[-1]
            _max_steps, pct, step, total, success = match.groups()
            result["rollout"] = f"step {step}/{total} ({pct}%), running success {success}%"
        batch_matches = list(re.finditer(
            r"Stepping through eval batches:\s+(\d+)%.*?\|\s*(\d+)/(?:\s*)?(\d+)",
            clean,
        ))
        if batch_matches:
            match = batch_matches[-1]
            pct, done, total = match.groups()
            result["batch"] = f"batch {done}/{total} ({pct}%)"
    except Exception as exc:
        result["parse_error"] = f"progress parser skipped: {type(exc).__name__}: {exc}"
    for line in reversed([line.strip() for line in clean.splitlines() if line.strip()]):
        if any(key in line for key in ["Running rollout", "Stepping through eval batches", "hf_access OK", "torch ", "cuda_available", "Overall Aggregated Metrics", "End of eval", "Saved results", "Traceback", "Error", "Exception"]):
            result["last"] = line[-260:]
            break
    return result


def activation_chunk_count() -> int:
    events = run_dir / "activation_capture" / "events.jsonl"
    if events.exists():
        try:
            data = tail_text(events, max_bytes=max(events.stat().st_size, 1))
            return data.count('"type":"chunk_start"')
        except OSError:
            pass
    image_dir = run_dir / "activation_capture" / "images"
    if image_dir.exists():
        chunks = set()
        for path in image_dir.glob("chunk_*_*"):
            parts = path.name.split("_")
            if len(parts) > 1:
                chunks.add(parts[1])
        return len(chunks)
    return 0


def print_eval_progress(start_time: float, final: bool = False, failed: bool = False):
    text = tail_text(launcher_log) + "\n" + tail_text(run_dir / "run.log")
    parsed = parse_progress_line(text)
    videos = list((run_dir / "videos").glob("**/*.mp4")) if (run_dir / "videos").exists() else []
    expected = expected_eval_count()
    video_pct = min(100.0, 100.0 * len(videos) / expected) if expected else 0.0
    chunks = activation_chunk_count() if CAPTURE_ACTIVATIONS else 0
    output_size = eval_format_bytes(eval_path_size_bytes(run_dir))
    parts = [
        f"elapsed {eval_format_duration(time.time() - start_time)}",
        f"videos {len(videos)}/{expected} ({video_pct:.1f}%)",
        f"output {output_size}",
    ]
    if CAPTURE_ACTIVATIONS:
        cap = int(CAPTURE_MAX_CHUNKS)
        parts.append(f"activation chunks {chunks}/{cap if cap > 0 else 'unlimited'}")
    if parsed.get("batch"):
        parts.append(parsed["batch"])
    if parsed.get("rollout"):
        parts.append(parsed["rollout"])
    prefix = "\n[eval stopped] " if failed else ("\n[eval complete] " if final else "\n[eval progress] ")
    print(prefix + " | ".join(parts), flush=True)
    if parsed.get("parse_error"):
        print("[eval parser] " + parsed["parse_error"], flush=True)
    if parsed.get("last"):
        print("[eval last] " + parsed["last"], flush=True)


def print_failure_diagnostics(returncode: int):
    print(f"\nEval failed with exit code {returncode}", flush=True)
    print(f"Run dir: {run_dir}", flush=True)
    print(f"Launcher log: {launcher_log}", flush=True)
    print(f"Eval log: {run_dir / 'run.log'}", flush=True)
    try:
        subprocess.run(["nvidia-smi"], check=False)
    except Exception as exc:
        print(f"Could not run nvidia-smi: {exc}", flush=True)
    for label, path in [("launcher log", launcher_log), ("eval run.log", run_dir / "run.log")]:
        if not path.exists():
            print(f"\n--- no {label} yet ---", flush=True)
            continue
        lines = [line for line in clean_log_text(tail_text(path, 80_000)).splitlines() if line.strip()]
        print(f"\n--- last {min(80, len(lines))} lines from {label} ---", flush=True)
        for line in lines[-80:]:
            print(line[-500:], flush=True)


print("Running:", " ".join(cmd), flush=True)
print("Run dir:", run_dir, flush=True)
eval_progress_seconds = max(5, int(globals().get("EVAL_PROGRESS_SECONDS", 30)))
print("Progress heartbeat seconds:", eval_progress_seconds, flush=True)
start = time.time()
with launcher_log.open("wb") as log_handle:
    process = subprocess.Popen(
        cmd,
        cwd=LOCAL_REPO,
        env=run_env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=0,
    )

    def write_stdout(chunk: bytes):
        buffer = getattr(sys.stdout, "buffer", None)
        if buffer is not None:
            buffer.write(chunk)
            buffer.flush()
        else:
            sys.stdout.write(chunk.decode("utf-8", errors="replace"))
            sys.stdout.flush()

    def stream_output():
        assert process.stdout is not None
        while True:
            chunk = process.stdout.read(4096)
            if not chunk:
                break
            write_stdout(chunk)
            log_handle.write(chunk)
            log_handle.flush()

    stream_thread = threading.Thread(target=stream_output, daemon=True)
    stream_thread.start()
    while process.poll() is None:
        time.sleep(eval_progress_seconds)
        if process.poll() is None:
            print_eval_progress(start)
    stream_thread.join(timeout=10)
    returncode = process.returncode

print_eval_progress(start, final=(returncode == 0), failed=(returncode != 0))
if returncode != 0:
    print_failure_diagnostics(returncode)
    raise subprocess.CalledProcessError(returncode, cmd)

LATEST_RUN = run_dir
print("Latest run:", LATEST_RUN)


In [ ]:
# @title Persist Outputs And Caches Back To Drive

# Outputs are small enough to sync as folders and should be visible directly in Drive.
rsync_tree(LOCAL_REPO / "outputs", DRIVE_OUTPUTS, reset=False)

# Persist caches as tar archives to avoid slow Drive small-file copies on the next Colab session.
if CACHE_TRANSFER_MODE == "archive":
    if FORCE_AUTH_REFRESH or not HF_HOME_ARCHIVE.exists():
        timed("write HF cache archive to Drive", lambda: create_tar(LOCAL_HF_HOME, HF_HOME_ARCHIVE))
    timed("write LIBERO cache archive to Drive", lambda: create_tar(LOCAL_LIBERO_CACHE, LIBERO_CACHE_ARCHIVE))
    timed("write LIBERO datasets archive to Drive", lambda: create_tar(LOCAL_DATA_ROOT, LIBERO_DATASETS_ARCHIVE))
else:
    timed("sync unpacked HF cache to Drive", lambda: rsync_tree(LOCAL_HF_HOME, DRIVE_HF_HOME))
    timed("sync unpacked LIBERO cache to Drive", lambda: rsync_tree(LOCAL_LIBERO_CACHE, DRIVE_LIBERO_CACHE))
    timed("sync unpacked LIBERO datasets to Drive", lambda: rsync_tree(LOCAL_DATA_ROOT, DRIVE_LIBERO_DATASETS))

print("Persisted outputs to:", DRIVE_OUTPUTS)
print("Cache archive dir:", DRIVE_ARCHIVES)

In [ ]:
# @title Display Summary, Videos, And Activation Report Inline

import json
import subprocess
from pathlib import Path
from IPython.display import display, Video, Markdown, Image, HTML

run_dir = LATEST_RUN
info_path = run_dir / "eval_info.json"
if info_path.exists():
    info = json.loads(info_path.read_text())
    overall = info.get("overall", {})
else:
    info = {}
    overall = {}

display(Markdown(f"""
### Latest run

`{run_dir}`

- success: `{overall.get('pc_success', 'n/a')}`
- episodes: `{overall.get('n_episodes', 'n/a')}`
- eval seconds: `{overall.get('eval_s', 'n/a')}`
"""))

videos = sorted((run_dir / "videos").glob("**/*.mp4"))
print("rollout videos:", len(videos))
if videos:
    display(Video(str(videos[0]), embed=True, width=720))

if CAPTURE_ACTIVATIONS:
    analysis_dir = run_dir / "analysis"
    analysis_dir.mkdir(parents=True, exist_ok=True)
    if GENERATE_DIAGNOSTIC_VIDEO:
        analysis_cmd = [
            str(PYTHON), "scripts/make_pi05_analysis_video.py",
            "--run", str(run_dir),
            "--task-id", str(ANALYSIS_TASK_ID),
            "--preview-frame", "30",
        ]
        subprocess.run(analysis_cmd, cwd=LOCAL_REPO, check=True)
        previews = sorted(analysis_dir.glob("*_frame0030.png"))
        analysis_videos = sorted(analysis_dir.glob("*.mp4"))
        if previews:
            display(Markdown("### Four-panel diagnostic preview"))
            display(Image(filename=str(previews[-1]), width=1000))
        if analysis_videos:
            display(Video(str(analysis_videos[-1]), embed=True, width=900))

    report_cmd = [
        str(PYTHON), "scripts/make_pi05_colab_report.py",
        "--run", str(run_dir),
        "--task-id", str(ANALYSIS_TASK_ID),
        "--episode", "0",
        "--n-action-steps", "10",
        "--max-rows", str(REPORT_MAX_ROWS),
    ]
    subprocess.run(report_cmd, cwd=LOCAL_REPO, check=True)
    report_dir = analysis_dir / f"task_{ANALYSIS_TASK_ID}_episode_0_colab_report"
    manifest_path = report_dir / f"task_{ANALYSIS_TASK_ID}_episode_0_manifest.json"
    manifest = json.loads(manifest_path.read_text())

    display(Markdown("""
### Granular Investigation Report

The interactive report lets you switch between activation family, metric, and layer without rendering every layer image inline. The chunk matrix still gives one row per policy call: simulator third-person frame, the two camera tensors fed to Pi0.5, first 10 actions, expert-layer mean, and expert layer-by-denoise activation.
"""))
    interactive_path = manifest.get("interactive_html")
    if interactive_path and Path(interactive_path).exists():
        display(HTML(Path(interactive_path).read_text()))
    else:
        for key, width in [
            ("chunk_matrix", 1600),
            ("family_heatmaps", 1100),
            ("expert_layers_grid", 1100),
        ]:
            path = manifest.get(key)
            if path and Path(path).exists():
                display(Markdown(f"#### {key.replace('_', ' ').title()}"))
                display(Image(filename=str(path), width=width))

    layer_graphs = [Path(p) for p in manifest.get("expert_layer_graphs", [])]
    if DISPLAY_INDIVIDUAL_LAYER_GRAPHS and layer_graphs:
        display(Markdown("#### Individual Expert Layer Graphs"))
        for path in layer_graphs[:max(0, int(LAYER_GRAPH_LIMIT))]:
            if path.exists():
                display(Image(filename=str(path), width=900))

    rsync_tree(LOCAL_REPO / "outputs", DRIVE_OUTPUTS, reset=False)
else:
    display(Markdown("Activation report skipped because `CAPTURE_ACTIVATIONS=False`."))


## Experimental: One-Chunk Plaintext Prompt Probe

This cell is for mechanistic interpretability, not benchmark scoring.

It resets one LIBERO task environment, captures the two camera views and robot state, replaces the task language with `PROBE_LANGUAGE`, and asks Pi0.5 for one 50-action chunk. It saves the input images and action chunk so you can compare plaintext prompts against internal activations and action outputs.

Caveat: Pi0.5-LIBERO was fine-tuned on LIBERO-style task prompts. Low-level prompts like `move +x` or `close gripper` may be out of distribution. Use this to inspect behavior, not to claim official benchmark success.

In [ ]:
# @title Run One-Chunk Plaintext Prompt Probe

import os
import subprocess
from pathlib import Path

probe_out = LOCAL_REPO / "outputs/probes" / PROBE_SUITE / f"task_{PROBE_TASK_ID}"
probe_out.mkdir(parents=True, exist_ok=True)

probe_code = r"""
import json
import os
from pathlib import Path

import cv2
import numpy as np
import torch

from lerobot.configs.policies import PreTrainedConfig
from lerobot.envs.configs import LiberoEnv
from lerobot.envs.factory import make_env, make_env_pre_post_processors
from lerobot.policies.factory import make_policy, make_pre_post_processors
from lerobot.scripts.lerobot_eval import preprocess_observation

suite = os.environ["PROBE_SUITE"]
task_id = int(os.environ["PROBE_TASK_ID"])
prompt = os.environ["PROBE_LANGUAGE"]
out_dir = Path(os.environ["PROBE_OUT"])
out_dir.mkdir(parents=True, exist_ok=True)

policy_cfg = PreTrainedConfig.from_pretrained(
    "lerobot/pi05_libero_finetuned",
    cache_dir=os.environ["HF_HUB_CACHE"],
    local_files_only=os.environ.get("HF_HUB_OFFLINE") == "1",
)
policy_cfg.device = "cuda"
policy_cfg.dtype = "bfloat16"
policy_cfg.compile_model = False
policy_cfg.gradient_checkpointing = False
policy_cfg.n_action_steps = 10

env_cfg = LiberoEnv(task=suite, task_ids=[task_id])
envs = make_env(env_cfg, n_envs=1, use_async_envs=False)
env = envs[suite][task_id]
policy = make_policy(cfg=policy_cfg, env_cfg=env_cfg, rename_map={})
policy.eval()

preprocessor, postprocessor = make_pre_post_processors(
    policy_cfg=policy_cfg,
    pretrained_path=policy_cfg.pretrained_path,
    preprocessor_overrides={
        "device_processor": {"device": str(policy.config.device)},
        "rename_observations_processor": {"rename_map": {}},
    },
)
env_preprocessor, _env_postprocessor = make_env_pre_post_processors(env_cfg=env_cfg, policy_cfg=policy_cfg)

obs, info = env.reset(seed=[int(os.environ.get("PROBE_SEED", "1000"))])
raw_render = env.envs[0].render() if hasattr(env, "envs") else env.call("render")[0]
cv2.imwrite(str(out_dir / "render.png"), cv2.cvtColor(raw_render, cv2.COLOR_RGB2BGR))

obs = preprocess_observation(obs)
obs["task"] = [prompt]
for key, value in obs.items():
    if key.startswith("observation.images."):
        arr = value[0]
        if hasattr(arr, "detach"):
            arr = arr.detach().cpu().numpy()
        if arr.shape[0] in (1, 3):
            arr = np.moveaxis(arr, 0, -1)
        if arr.max() <= 1.5:
            arr = np.clip(arr, 0, 1) * 255
        cv2.imwrite(str(out_dir / f"{key.replace('.', '_')}.png"), cv2.cvtColor(arr.astype(np.uint8), cv2.COLOR_RGB2BGR))

batch = env_preprocessor(obs)
batch = preprocessor(batch)
with torch.inference_mode():
    actions = policy.predict_action_chunk(batch).detach().cpu().float()[0]

payload = {
    "suite": suite,
    "task_id": task_id,
    "prompt": prompt,
    "action_shape": list(actions.shape),
    "first_10_actions": actions[:10].tolist(),
    "mean_abs_per_dim": actions.abs().mean(dim=0).tolist(),
}
(out_dir / "action_chunk.json").write_text(json.dumps(payload, indent=2))
print(json.dumps(payload, indent=2)[:4000])
env.close()
"""

env = os.environ.copy()
env.update({
    "PROBE_SUITE": PROBE_SUITE,
    "PROBE_TASK_ID": str(PROBE_TASK_ID),
    "PROBE_LANGUAGE": PROBE_LANGUAGE,
    "PROBE_SEED": str(PROBE_SEED),
    "PROBE_OUT": str(probe_out),
    "MPLBACKEND": "Agg",
})
subprocess.run([str(PYTHON), "-c", probe_code], cwd=LOCAL_REPO, env=env, check=True)
rsync(LOCAL_REPO / "outputs", DRIVE_OUTPUTS)
print("Probe saved to:", probe_out)

## Run All Checklist

Before sharing:

1. Save a copy of this notebook in the restricted Drive folder.
2. Share the notebook only with `programmer908@gmail.com` and other approved accounts.
3. Share `DRIVE_ROOT` only with the same approved accounts.
4. For first cache fill, keep `CACHE_TRANSFER_MODE="archive"`, set `ALLOW_AUTH_REFRESH=True`, and store `HF_TOKEN` only in private Colab Secrets.
5. After `archives/hf_home.tar` exists, leave `ALLOW_AUTH_REFRESH=True` or set it to `False`; the archive path is used first unless `FORCE_AUTH_REFRESH=True`.
6. Set Runtime -> Change runtime type -> GPU -> L4 when available. Keep `REQUIRED_GPU="Any"` if Colab gives you a T4 and you still want a smoke test.
7. Runtime -> Run all.

Outputs are persisted back to `DRIVE_ROOT/outputs` after every run, including rollout videos, diagnostic videos, chunk matrices, activation heatmaps, and individual layer graphs. Model/assets caches are persisted as tar archives under `DRIVE_ROOT/archives` so future sessions start faster.